# Simple visualisation of CYGNSS data (inundation observations, interpolated inundation map, inundation anomalies, interpolated inundation anomalies map) for given date


Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import geemap
import ee
import cmocean
import matplotlib as mpl
import numpy as np

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")

Set the date of interesrt (doy and year) and filter the appropriate CYGNSS data

In [4]:
doy = 311
year = 2025

image = cygnss_collection.filter(
    ee.Filter.And(
        ee.Filter.eq('year', year),
        ee.Filter.eq('doy', doy)
    )
).first()


Set up the functions to (i) remove NaN values from CYGNSS data, (ii) convert inundation data to fractional inundation, and (iii) remove bias from the fractional inundation data (in case of cygnss anomalies).

In [5]:
def mask_inundation(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    return image.divide(100)  # scale to fractional values

def mask_anomalies(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    image = image.subtract(100)               # remove bias
    return image.divide(100)                  # scale to fractional values

Select from the Image the bands of interest: inundation_1day, inundation_3day, interpolation_flag, inundation_anomaly_1day or inundation_anomaly_3day.

In [6]:
inundation_observations = mask_inundation(image.select('inundation_1day')) # selecting the appropriate band and applying a function to mask no-data values and scale to fractional inundation
inundation_map = mask_inundation(image.select('inundation_3day')) #as above
inundation_map_anomalies = mask_anomalies(image.select('inundation_anomaly_3day')) # selecting the appropriate band and applying a function to mask no-data values, removing bias, and scale to fractional inundation

Set the visualization parameters (colors and value ranges) for the plotted bands. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. The code below converts Matplotlib colormaps to HEX color codes. Separate visualization parameters are defined for inundation data (nominal range: 0 to 1) and inundation anomaly data (nominal range: −1 to 1).

In [7]:
cmap_inundation = mpl.colormaps["gist_earth_r"].resampled(256) # palette (gist_earth) from matplotlib (https://matplotlib.org/stable/users/explain/colors/colormaps.html)
palette_inundation_hex = [mpl.colors.rgb2hex(cmap_inundation(i)) for i in range(256)]
inundation_vis = {
    'min': 0,
    'max': 1,
    'palette': palette_inundation_hex
}


anomaly_palette = cmocean.cm.balance_r(np.linspace(0, 1, 256)) #palette (balance) from matplotlib (https://matplotlib.org/cmocean/)
anomaly_palette_hex = [mpl.colors.rgb2hex(color) for color in anomaly_palette]
anomaly_vis = {
    'min': -0.2,  
    'max': 0.2,  
    'palette': anomaly_palette_hex  
}

Plot interpolated inundation map (band 'inundation_3day'). 

In [8]:
# Region of interest – South America
roi = ee.Geometry.Rectangle([
    -82, -56,   # west, south
    -34, 13     # east, north
])

Map = geemap.Map()# generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
Map.addLayer(inundation_map, inundation_vis, "Interpolated inundation map") # Add the inundation anomaly layer using the inundation_vis parameters.
Map.add_colorbar(vis_params=inundation_vis, label="Fractional inundation", font_size=12)  # Add colorbar
Map.center_object(roi, 4) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[-19.28693290705331, -58.00000000000001], controls=(WidgetControl(options=['position', 'transparent…

Plot interpolated inundation map anomalies (band 'inundation_anomaly_3day')

In [9]:
# Region of interest – South America
roi = ee.Geometry.Rectangle([
    -82, -56,   # west, south
    -34, 13     # east, north
])
Map = geemap.Map() #generation of map
Map.add_basemap('HYBRID')#adding 'HYBRID' layer that contain satellite world imaginery
Map.addLayer(inundation_map_anomalies, anomaly_vis,"Interpolated anomaly map")# Adding layer with inundation anomalies, that willbe displayedusing inundation_vis params
Map.add_colorbar(vis_params=anomaly_vis, label="Fractional inundation anomaly") # Adding colorbar
Map.center_object(roi, 4) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[-19.28693290705331, -58.00000000000001], controls=(WidgetControl(options=['position', 'transparent…

Plot inundation observations (band 'inundation_1day'). Please note that when plotting observational data (which contains a large number of no-data pixels), it may be necessary to zoom in to properly visualize the colors.

In [12]:
#Prepare Sentinel Imagefor ROI
def mask_s2_clouds(image):
    scl = image.select('SCL')
    mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(mask).select(['B4', 'B3', 'B2'])

# Cloud-masked median composite for 1–30 November 2025
sentinel2_rgb = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2025-11-01', '2025-12-01')
    .map(mask_s2_clouds)
    .median()
    .clip(roi)
)

# RGB display settings
sentinel2_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.1
}

In [15]:
# Region of interest – north-eastern Botswana
roi = ee.Geometry.Rectangle([
    25, -22,   # west, south
    27, -19     # east, north
])
Map = geemap.Map() # map generationp
#Map.add_basemap('HYBRID')# add the 'HYBRID' layer containing satellite imagery
Map.addLayer(sentinel2_rgb, sentinel2_vis, 'Sentinel-2 RGB median composite (Nov 2025)') #Add Sentinel-2 RGB composite
Map.addLayer(inundation_observations, inundation_vis, "Fractional inundation" )# add the inundation anomaly layer using the inundation_vis parameters
Map.add_colorbar(vis_params=inundation_vis, label="Fractional inundation") # Add colorbar
Map.center_object(roi, 8) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[-20.49796427094952, 25.999999999999908], controls=(WidgetControl(options=['position', 'transparent…

Plot inundation observations anomalies (band 'inundation_anomaly_1day'). Please note that when plotting observational data (which contains a large number of no-data pixels), it may be necessary to zoom in to properly visualize the colors.

In [14]:
# Region of interest – Semarang/ Central Java
roi = ee.Geometry.Rectangle([
    110, -7.5,   # west, south
    111, -6.5     # east, north
])
doy = 295
year = 2025

image = cygnss_collection.filter(
    ee.Filter.And(
        ee.Filter.eq('year', year),
        ee.Filter.eq('doy', doy)
    )
).first()

inundation_anomalies = mask_anomalies(image.select('inundation_anomaly_1day')) # selecting the appropriate band and applying a function to mask no-data values, removing bias, and scale to fractional inundation

Map = geemap.Map() #generation of map
#Map.add_basemap('HYBRID')#adding 'HYBRID' layer that contain satellite world imaginery
Map.addLayer(inundation_anomalies, anomaly_vis, "Inundation anomalies")# Adding layer with inundation anomalies, that willbe displayedusing inundation_vis params
Map.add_colorbar(vis_params=anomaly_vis, label="Fractional inundation anomaly",font_size=12) # Adding colorbar
Map.center_object(roi, 11) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.

Map(center=[-7.000085277029306, 110.50000000000033], controls=(WidgetControl(options=['position', 'transparent…